# RSNA Knee Abnormality Detection — Exploratory Data Analysis

**Competition:** [RSNA Knee Abnormality Detection](https://www.kaggle.com/competitions/rsna-knee-abnormality-detection)
**Task:** predict twelve knee-MRI abnormalities per study.
**Metric:** macro-averaged ROC AUC over the twelve labels.
**Constraints:** notebooks only, 9 hours, no internet, final submission October 22, 2026.

---

## TL;DR — what this notebook establishes

1. **Only 58 of 4,407 training studies are labeled**, and labeling is strictly all-or-nothing.
   The other 4,349 have a radiology report and nothing else. → [Section 2](#2-cohorts)
2. **The gold 58 cannot select models.** Per-label AUC standard errors at that sample size are
   large enough to swamp genuine differences between candidates, and under resampling several
   labels lose all their positives or negatives entirely. Treat gold as an audit set for a label
   extractor. → [Section 3](#3-the-metric-and-the-evaluation-noise-floor)
3. **Reports are a training signal, not an inference input.** `test.csv` has one column. Any
   design that reads report text at prediction time fails the rerun. → [Section 1](#1-inventory-and-schema-contract)
4. **Reports are multilingual**, so weak labeling is a multilingual NLP problem with negation,
   not a keyword grep. → [Section 4](#4-the-reports)
5. **A transparent rule-based extractor already recovers a usable fraction of the gold labels**
   for the tractable findings, and the audit shows exactly where it fails.
   → [Section 5](#5-weak-label-baseline-and-gold-audit)
6. **`Fluid_Sensitive` and `Fat_Suppression` are the same column**, identical in all 24,371 rows.
   → [Section 6](#6-series-structure-and-protocol)
7. **The 9-hour budget is the binding engineering constraint**, leaving roughly 25 seconds per
   test study including DICOM decode. → [Section 11](#11-inference-budget)

## How to run this

The notebook detects its environment and runs in one of two modes.

| Mode | Trigger | What runs |
| --- | --- | --- |
| **CSV-only** | no `train_series/` directory, or `pydicom` unavailable | Everything except sections 7 and 9, which print why they are skipping and continue |
| **Full** | DICOM tree present and `pydicom` importable (Kaggle) | Adds the header census, plane verification, transfer-syntax audit, pixel montages, and patient-level grouped folds |

Hard dependencies are **pandas, numpy and matplotlib** only. `seaborn` is used for styling if it
imports and skipped otherwise. Grouped folds, Fisher exact tests and Wilson intervals are
implemented here rather than imported, so nothing needs `sklearn` or `scipy`.

## Contents

| Section | Question it answers |
| --- | --- |
| [1. Inventory and schema contract](#1-inventory-and-schema-contract) | What files exist, what is in them, and is the data internally consistent? |
| [2. Cohorts](#2-cohorts) | How many studies are actually labeled, and what do those labels look like? |
| [3. Metric and noise floor](#3-the-metric-and-the-evaluation-noise-floor) | Can 58 studies rank models? Which labels drive the score? |
| [4. The reports](#4-the-reports) | What language are they in, how long, and are any duplicated? |
| [5. Weak-label baseline](#5-weak-label-baseline-and-gold-audit) | How far does a transparent rule-based extractor get? |
| [6. Series and protocol](#6-series-structure-and-protocol) | What does a study physically contain? |
| [7. DICOM header census](#7-dicom-header-census) | What scanners, geometries and transfer syntaxes? |
| [8. Confounding](#8-confounding) | Is the labeled subset systematically different from the rest? |
| [9. Pixels](#9-pixels) | What do the images look like and how should they be normalized? |
| [10. Splits](#10-splits) | How do we build a validation split that does not leak? |
| [11. Inference budget](#11-inference-budget) | Does a given pipeline fit in 9 hours? |
| [12. Submission validator](#12-submission-validator) | Is our submission well-formed? |
| [13. Findings](#13-findings) | What did we learn and what should we do next? |

## Clinical glossary

The twelve labels are radiology terms, and the anatomy explains several statistical patterns we
will see later, so it is worth two minutes up front. They fall into four families.

**Ligaments** — bands connecting bone to bone, providing joint stability.

| Label | What it is | Usually read on |
| --- | --- | --- |
| `ACL` | Anterior cruciate ligament injury; the central stabilizer, the classic pivot injury | Sagittal, fluid-sensitive |
| `MCL` | Medial collateral ligament injury; the inner-side stabilizer | Coronal, fluid-sensitive |

**Menisci** — C-shaped cartilage pads that cushion the joint.

| Label | What it is | Usually read on |
| --- | --- | --- |
| `Medial Meniscus` | Tear of the inner pad | Sagittal + coronal |
| `Lateral Meniscus` | Tear of the outer pad | Sagittal + coronal |

**Osteoarthritis, by compartment** — cartilage loss and degeneration, localized to one of three
compartments of the knee.

| Label | What it is | Usually read on |
| --- | --- | --- |
| `Medial OA` | Inner tibiofemoral compartment | Coronal |
| `Lateral OA` | Outer tibiofemoral compartment | Coronal |
| `PF OA` | Patellofemoral: kneecap against the femoral groove | Axial |

**Fluid, inflammation and bone**

| Label | What it is | Usually read on |
| --- | --- | --- |
| `Effusion` | Excess fluid inside the joint capsule | Fluid-sensitive, any plane |
| `Synovitis` | Inflammation of the joint lining | Fluid-sensitive |
| `Baker's` | Fluid collection bulging behind the knee | Axial/sagittal, fluid-sensitive |
| `Contusion` | Bone bruise — marrow edema without a fracture line | Fluid-sensitive + fat-suppressed |
| `Fracture` | Break in the bone | Fluid-sensitive + fat-suppressed |

**Sequence and geometry terms**

- **Fluid-sensitive** (T2, PD, STIR): fluid appears bright. Effusion, synovitis and Baker's cyst
  are essentially defined by bright fluid, which is why they co-occur for physical reasons and not
  merely statistical ones.
- **Fat suppression**: fat signal is nulled so that marrow edema stands out. This is what makes
  contusion and fracture visible at all.
- **Planes**: *sagittal* slices side-to-side (best for cruciate ligaments and menisci), *coronal*
  front-to-back (best for collateral ligaments and the medial/lateral compartments), *axial*
  top-to-bottom (best for the patellofemoral joint and Baker's cysts).

**Why the grouping matters for modeling.** Seven of the twelve labels are medial/lateral pairs. A
horizontal flip augmentation that does not also swap `Medial Meniscus` with `Lateral Meniscus` and
`Medial OA` with `Lateral OA` will train the model on inverted targets. That is a correctness bug,
not a tuning preference.

## 0. Configuration

Everything environment-dependent is decided once, here: imports, the single random seed, sampling
budgets, where the data lives, and whether DICOMs are reachable. Later sections read these values
and never redefine them, so there is exactly one place to change behaviour.

In [ ]:
import json
import math
import os
import re
import sys
import unicodedata
from collections import Counter, defaultdict
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt

# Optional, styling only. The notebook must look fine without it.
try:
    import seaborn as sns
    HAS_SEABORN = True
    sns.set_theme(style="whitegrid", palette="muted")
except ImportError:
    HAS_SEABORN = False
    plt.style.use("ggplot")

# Optional, needed only for the DICOM sections.
try:
    import pydicom
    HAS_PYDICOM = True
except ImportError:
    HAS_PYDICOM = False

SEED = 20260805
RNG = np.random.default_rng(SEED)

# Sampling budgets. Header/pixel reads are I/O bound on Kaggle, so they are capped rather than
# exhaustive; every figure derived from them is reported as a sample estimate.
N_HEADER_SERIES = 400      # series to open for the header census
N_PIXEL_SERIES = 6         # series to render in the montage
N_BOOTSTRAP = 2000         # bootstrap resamples for the noise floor
N_FOLDS = 5

# Competition constants, from the overview and dataset description.
RUNTIME_LIMIT_SECONDS = 9 * 60 * 60      # 32,400
N_TEST_STUDIES_DOCUMENTED = 1300         # "about 1300 studies in the test set"
MEDIAN_SLICES_PER_SERIES = 30

REQUIRED_FILES = [
    "train.csv", "train_series.csv", "test.csv", "test_series.csv", "sample_submission.csv",
]


def resolve_data_root():
    """Return the first directory containing all five competition CSVs.

    Checks the Kaggle mount, then common local layouts. Raises rather than guessing, because a
    silent wrong root produces confusing downstream errors.
    """
    candidates = [
        Path("/kaggle/input/competitions/rsna-knee-abnormality-detection"),
        Path("input"),
        Path("data"),
        Path("."),
        Path(".."),
    ]
    kaggle_input = Path("/kaggle/input")
    if kaggle_input.is_dir():
        candidates.extend(sorted(kaggle_input.iterdir()))

    for root in candidates:
        try:
            if all((root / name).exists() for name in REQUIRED_FILES):
                return root.resolve()
        except OSError:
            continue
    raise FileNotFoundError(
        "Could not locate the competition CSVs. Tried: "
        + ", ".join(str(c) for c in candidates)
    )


DATA_ROOT = resolve_data_root()
TRAIN_DICOM_DIR = DATA_ROOT / "train_series"


def survey_dicom_tree(root, probe_limit=64):
    """Map study UID -> directory for studies that actually have series on disk.

    A directory existing is not enough. A partially downloaded or partially extracted tree can
    contain empty study folders, which would make the DICOM sections run on nothing and report
    empty results as if they were findings. Only the first `probe_limit` studies are probed for
    series, which is enough to tell "populated" from "empty" without walking 800k files.
    """
    if not root.is_dir():
        return {}, 0
    study_dirs = {}
    try:
        entries = sorted(entry for entry in root.iterdir() if entry.is_dir())
    except OSError:
        return {}, 0

    populated = 0
    for index, entry in enumerate(entries):
        study_dirs[entry.name] = entry
        if index < probe_limit:
            try:
                if any(child.is_dir() for child in entry.iterdir()):
                    populated += 1
            except OSError:
                pass
    return study_dirs, populated


DICOM_STUDY_DIRS, DICOM_POPULATED_PROBE = survey_dicom_tree(TRAIN_DICOM_DIR)
HAS_DICOM = HAS_PYDICOM and DICOM_POPULATED_PROBE > 0

ARTIFACT_DIR = Path("eda_artifacts")
ARTIFACT_DIR.mkdir(exist_ok=True)

pd.set_option("display.width", 120)
pd.set_option("display.max_columns", 40)

print("Environment")
print("-" * 64)
print(f"  python            {sys.version.split()[0]}")
print(f"  pandas            {pd.__version__}")
print(f"  numpy             {np.__version__}")
print(f"  matplotlib        {matplotlib.__version__}")
print(f"  seaborn           {'yes' if HAS_SEABORN else 'no (matplotlib styling)'}")
print(f"  pydicom           {'yes' if HAS_PYDICOM else 'no'}")
print()
print(f"  data root         {DATA_ROOT}")
print(f"  artifacts         {ARTIFACT_DIR.resolve()}")
print(f"  seed              {SEED}")
print()

if not TRAIN_DICOM_DIR.is_dir():
    print("  DICOM tree        absent")
else:
    print(f"  DICOM tree        {TRAIN_DICOM_DIR}")
    print(f"                    {len(DICOM_STUDY_DIRS):,} study folders, "
          f"{DICOM_POPULATED_PROBE} of the first 64 contain series")

print(f"  mode              {'FULL (headers + pixels)' if HAS_DICOM else 'CSV-ONLY (sections 7 and 9 skip)'}")
if not HAS_DICOM:
    reason = "pydicom not installed" if not HAS_PYDICOM else "no populated study folders found"
    print(f"                    reason: {reason}")

## 1. Inventory and schema contract

Before any analysis, we pin down the contract: which files exist, what columns they carry, and
whether the pieces refer to each other consistently. Two questions here are worth more than the
rest of the section combined.

**Are the label names exactly what the scorer expects?** Submission columns include an apostrophe
(`Baker's`) and embedded spaces (`Medial Meniscus`, `PF OA`). Retyping them by hand is a reliable
way to produce a silently misaligned submission, so we assert against `sample_submission.csv` and
against the order published in the competition overview.

**Are the radiology reports available at test time?** The competition describes a dataset that
"pairs every imaging study with its original radiology report", which sounds like a multimodal
problem where text is an input. If reports were available at inference, the optimal solution would
look completely different — you would parse text rather than pixels. This is the single most
consequential structural question in the competition, so we settle it with a schema assertion
rather than an opinion.

In [ ]:
train = pd.read_csv(DATA_ROOT / "train.csv")
train_series = pd.read_csv(DATA_ROOT / "train_series.csv")
test = pd.read_csv(DATA_ROOT / "test.csv")
test_series = pd.read_csv(DATA_ROOT / "test_series.csv")
sample_submission = pd.read_csv(DATA_ROOT / "sample_submission.csv")

# The order published in the competition overview's submission format.
EXPECTED_LABELS = [
    "ACL", "MCL", "Medial Meniscus", "Lateral Meniscus",
    "Medial OA", "Lateral OA", "PF OA",
    "Effusion", "Synovitis", "Baker's", "Contusion", "Fracture",
]
LABELS = [c for c in sample_submission.columns if c != "StudyInstanceUID"]

assert LABELS == EXPECTED_LABELS, f"label mismatch:\n  file: {LABELS}\n  spec: {EXPECTED_LABELS}"
assert sample_submission.columns[0] == "StudyInstanceUID"
assert all(c in train.columns for c in LABELS), "train.csv is missing label columns"

for name, frame in [("train", train), ("train_series", train_series),
                    ("test", test), ("test_series", test_series),
                    ("sample_submission", sample_submission)]:
    print(f"  {name:20s} {frame.shape[0]:>7,} rows x {frame.shape[1]:>2} cols")

print(f"\nLabel names verified against sample_submission.csv and the published order ({len(LABELS)} labels).")

In [ ]:
# The decisive check: which columns exist in train but not in test?
train_only = [c for c in train.columns if c not in test.columns]
optional_columns = ["PatientSex"]

print("Columns in train.csv but NOT in test.csv:")
for col in train_only:
    kind = "label" if col in LABELS else "FEATURE"
    print(f"  [{kind:>7s}] {col}")

print(f"\ntest.csv columns: {list(test.columns)}")
print(f"Report column present at test time: {'Report' in test.columns}")

print("\nOptional columns documented by the host:")
for col in optional_columns:
    print(f"  {col:15s} train: {col in train.columns!s:5s}  test: {col in test.columns!s:5s}")

print(f"\ntest.csv holds {len(test):,} studies locally; the host documents ~{N_TEST_STUDIES_DOCUMENTED:,} at scoring.")
print("No test-side statistic in this notebook is generalizable.")

### Finding: reports are training-time only, and the local test file is a placeholder

`test.csv` has exactly one column, `StudyInstanceUID`. The `Report` column exists only in
training. Read against the official wording — reports let models *"learn from* both visual scans
and written diagnostic text" — the intent is clear: **reports are a supervision signal for
building labels, not a model input.**

This determines the shape of any viable solution. The reports are how you obtain labels for the
4,349 unlabeled studies; the model that consumes them must be image-only at inference.

Two secondary notes. `PatientSex` is documented by the host but is **absent from this copy** of
`train.csv`, so every use of it must be guarded by a presence check. And the local `test.csv`
carries 3 placeholder studies against roughly 1,300 at scoring, so nothing measured on the test
side generalizes — it is useful only for validating submission mechanics.

### Referential integrity

Three ways this dataset could be internally inconsistent, each of which would quietly corrupt
later analysis: duplicated study rows (double-counting), series pointing at studies that do not
exist (orphans), and studies with no series at all (nothing to predict from).

In [ ]:
def check_integrity(study_frame, series_frame, name):
    """Report duplicate study rows, orphan series, and studies with no imaging."""
    dup_studies = study_frame["StudyInstanceUID"].duplicated().sum()
    dup_series = series_frame["SeriesInstanceUID"].duplicated().sum()

    study_ids = set(study_frame["StudyInstanceUID"])
    series_study_ids = set(series_frame["StudyInstanceUID"])
    orphan_series = series_study_ids - study_ids
    studies_without_series = study_ids - series_study_ids

    print(f"{name}")
    print(f"  duplicate study rows          {dup_studies:,}")
    print(f"  duplicate series UIDs         {dup_series:,}")
    print(f"  series with unknown study     {len(orphan_series):,}")
    print(f"  studies with no series        {len(studies_without_series):,}")
    print(f"  studies / series              {len(study_ids):,} / {len(series_frame):,}"
          f"  ({len(series_frame) / max(len(study_ids), 1):.2f} series per study)")
    return {"orphan_series": orphan_series, "studies_without_series": studies_without_series}


train_integrity = check_integrity(train, train_series, "TRAIN")
print()
test_integrity = check_integrity(test, test_series, "TEST (placeholder)")

### Finding: the CSVs are internally consistent

No duplicate study rows, no duplicate series identifiers, no orphan series, and no study missing
its imaging. Every training study has at least one series, averaging 5.53 series per study. We can
proceed without defensive filtering, and any duplication we find later (section 4 finds some in
the report text) is a content-level issue rather than a structural one.

## 2. Cohorts

This is where the dataset's defining property shows up, and where the most common analysis
mistake happens.

The naive move is `train[LABELS].mean()` to get prevalence. That returns a number, the number
looks plausible, and it is wrong in a specific way: pandas skips `NaN`, so the result describes
only the labeled studies while appearing to describe all 4,407. The companion mistake,
`train[LABELS].sum(axis=1)`, counts unlabeled studies as studies with zero findings, manufacturing
thousands of fake healthy knees.

So the first thing to establish is the *structure* of missingness — is a study labeled for all
twelve findings, or can it be labeled for some? — and then define the cohorts once and use them
everywhere.

In [ ]:
label_counts_per_row = train[LABELS].notna().sum(axis=1)
distinct_counts = sorted(label_counts_per_row.unique().tolist())

print(f"Distinct counts of non-null labels per study: {distinct_counts}")
print(f"All-or-nothing labeling: {set(distinct_counts) <= {0, len(LABELS)}}")
print()

for count in distinct_counts:
    n = int((label_counts_per_row == count).sum())
    print(f"  {n:>6,} studies have {count:>2} of {len(LABELS)} labels populated")

GOLD_MASK = label_counts_per_row == len(LABELS)
WEAK_MASK = label_counts_per_row == 0

gold = train.loc[GOLD_MASK].copy()
weak = train.loc[WEAK_MASK].copy()
gold[LABELS] = gold[LABELS].astype(int)

N_GOLD, N_WEAK, N_TRAIN = len(gold), len(weak), len(train)
assert N_GOLD + N_WEAK == N_TRAIN, "partially labeled studies exist; the cohort logic needs revisiting"

print()
print(f"  gold cohort (fully labeled)   {N_GOLD:>6,}  ({N_GOLD / N_TRAIN:6.2%} of training)")
print(f"  weak cohort (report only)     {N_WEAK:>6,}  ({N_WEAK / N_TRAIN:6.2%} of training)")

### Finding: 58 labeled studies, 4,349 unlabeled, nothing in between

Labeling is strictly all-or-nothing — every study has either all twelve labels or none, with zero
partial rows. **1.32% of the training set carries ground truth.**

This is the fact that dictates the solution architecture. A purely supervised model has 58
examples to learn twelve findings from, which is not a viable training set. The 4,349 report-only
studies are the actual training data, reachable only through the text. Everything downstream
follows from this: sections 4 and 5 build the text pathway, and section 3 establishes what the
58 can and cannot legitimately be used for.

From here on, `gold` (n=58) and `weak` (n=4,349) are used explicitly, and **every statistic names
its cohort**.

In [ ]:
prevalence = pd.DataFrame({
    "positives": gold[LABELS].sum().astype(int),
    "negatives": (len(gold) - gold[LABELS].sum()).astype(int),
})
prevalence["prevalence"] = prevalence["positives"] / len(gold)
prevalence = prevalence.sort_values("prevalence", ascending=False)

print(f"Label prevalence - GOLD COHORT ONLY (n={len(gold)})")
print("-" * 58)
for label, row in prevalence.iterrows():
    bar = "#" * int(round(row["prevalence"] * 40))
    print(f"  {label:18s} {int(row['positives']):>3}/{len(gold)}  {row['prevalence']:6.1%}  {bar}")

positives_per_study = gold[LABELS].sum(axis=1)
print()
print(f"Findings per study (gold, n={len(gold)}): "
      f"min {positives_per_study.min()}, median {positives_per_study.median():.0f}, max {positives_per_study.max()}")
print(f"Studies with zero findings: {int((positives_per_study == 0).sum())}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

order = prevalence.index.tolist()
axes[0].barh(order[::-1], prevalence["prevalence"].values[::-1] * 100, color="#4c72b0")
axes[0].set_xlabel("Prevalence (%)")
axes[0].set_title(f"Effusion is the most common finding; MCL the rarest (gold, n={len(gold)})")
for i, value in enumerate(prevalence["prevalence"].values[::-1] * 100):
    axes[0].text(value + 0.8, i, f"{value:.1f}%", va="center", fontsize=9)
axes[0].set_xlim(0, 72)

counts = positives_per_study.value_counts().sort_index()
axes[1].bar(counts.index.astype(int), counts.values, color="#dd8452")
axes[1].set_xlabel("Number of positive findings in a study")
axes[1].set_ylabel("Studies")
axes[1].set_title(f"Every labeled study has at least one finding (gold, n={len(gold)})")
axes[1].set_xticks(range(0, int(counts.index.max()) + 1))

fig.tight_layout()
plt.show()

print(f"Caption: both panels describe the {len(gold)} fully labeled studies only, "
      f"which is {len(gold) / len(train):.2%} of the training set.")

### Finding: the gold cohort is finding-enriched, so its prevalence is not a corpus estimate

Prevalence on the gold 58 runs from **15.5% (MCL)** to **60.3% (Effusion)**, and **no labeled
study has zero findings** — the count of positives per study runs from 1 to 9. An unselected
clinical population would contain plenty of normal knees, so this cohort was clearly selected to
be informative rather than representative.

Two consequences, and both point the same way. First, these percentages describe 58 hand-picked
studies and must not be used as priors for the corpus or the test set. Second, the host explicitly
warns that prevalence is *not* guaranteed to match across training, public leaderboard and private
test. Together they rule out prevalence-matching, prior rescaling, or any calibration that assumes
a known base rate. Since the metric is AUC, which cares only about ranking, this costs us nothing.

### Which findings travel together?

Co-occurrence matters for two practical reasons. If labels are strongly dependent, independent
per-label heads leave information on the table and a multi-task or structured head should help.
And when we bootstrap in the next section, resampling has to be done over *studies* so that these
dependencies survive — resampling labels independently would produce a fantasy dataset.

We use the Jaccard index (intersection over union) rather than raw co-occurrence counts, so that
common labels do not dominate purely by being common.

In [ ]:
def jaccard_matrix(binary_frame):
    """Pairwise Jaccard index between binary label columns."""
    values = binary_frame.to_numpy(dtype=float)
    intersection = values.T @ values
    totals = values.sum(axis=0)
    union = totals[:, None] + totals[None, :] - intersection
    with np.errstate(divide="ignore", invalid="ignore"):
        result = np.where(union > 0, intersection / union, 0.0)
    return pd.DataFrame(result, index=binary_frame.columns, columns=binary_frame.columns)


jaccard = jaccard_matrix(gold[LABELS])

fig, ax = plt.subplots(figsize=(9, 7.5))
masked = jaccard.to_numpy().copy()
np.fill_diagonal(masked, np.nan)
image = ax.imshow(masked, cmap="viridis", vmin=0, vmax=np.nanmax(masked))
ax.set_xticks(range(len(LABELS)))
ax.set_xticklabels(LABELS, rotation=45, ha="right")
ax.set_yticks(range(len(LABELS)))
ax.set_yticklabels(LABELS)
ax.set_title(f"Label co-occurrence, Jaccard index (gold, n={len(gold)})")
fig.colorbar(image, ax=ax, fraction=0.046, label="Jaccard index")

for i in range(len(LABELS)):
    for j in range(len(LABELS)):
        if i != j and masked[i, j] >= 0.20:
            ax.text(j, i, f"{masked[i, j]:.2f}", ha="center", va="center",
                    color="white", fontsize=7.5)

fig.tight_layout()
plt.show()

pairs = [
    (a, b, jaccard.loc[a, b])
    for i, a in enumerate(LABELS) for b in LABELS[i + 1:]
]
pairs.sort(key=lambda item: item[2], reverse=True)
print(f"Strongest co-occurring pairs (gold, n={len(gold)}):")
for a, b, value in pairs[:6]:
    print(f"  {a:18s} + {b:18s}  Jaccard {value:.3f}")

### Finding: labels are correlated, and the correlations are clinically recognizable

The strongest pair is **Effusion + Synovitis (0.55)**, which is close to a tautology: an inflamed
joint lining produces fluid, and both are read as bright signal on the same fluid-sensitive
sequences.

More interesting is **ACL + Contusion (0.43)**. That is the pivot-shift injury pattern — when the
ACL ruptures, the femur and tibia strike each other and leave a bone bruise in a characteristic
location. The model is being asked to detect two consequences of one event, so the labels are not
just statistically associated but causally linked.

The degenerative cluster behaves the same way: **Medial OA + Baker's (0.42)** and **Medial
Meniscus + Medial OA (0.41)** are the medial-compartment degeneration story, where cartilage loss,
meniscal damage and a fluid-filled cyst travel together in the older knee. Note the contrast with
the ACL group, which is the acute-injury story in the younger knee. There appear to be roughly two
patient populations mixed in this dataset.

Two consequences. **Modeling:** independent binary heads ignore all of this, so multi-task heads on
a shared trunk are the natural starting point, and the fluid-sensitive series is the single highest
value input for at least four labels. **Methodology:** because labels travel together, the
bootstrap in section 3 resamples whole studies rather than labels.

## 3. The metric, and the evaluation noise floor

It is worth starting from the scoring rule rather than the data, because the rule has consequences
that are easy to miss.

The score is the **macro average of twelve per-label ROC AUCs**. Three things follow directly:

1. **Every label carries weight 1/12 no matter how rare it is.** The precision of an AUC estimate
   is governed by the *smaller* of the positive and negative counts, so the rarest labels are both
   the hardest to estimate and equally weighted. They inject the most variance into the final
   score.
2. **AUC is threshold-free.** Tuning a decision threshold, optimizing F1, or applying class weights
   to "fix" imbalance changes nothing. Only the ranking within each label matters.
3. **Each label is ranked independently.** Any monotone rescaling of one prediction column leaves
   the score untouched, so per-label rank-averaging across models is free.

The question this section answers: **can the 58 labeled studies be used to choose between two
models?** We answer it two ways — an analytic standard error per label, and a bootstrap that asks
how often a label becomes unscoreable altogether.

In [ ]:
def hanley_mcneil_se(auc, n_pos, n_neg):
    """Standard error of an AUC estimate (Hanley and McNeil, 1982).

    Assumes an exponential score distribution, which is the standard convenience assumption; it is
    accurate enough for sizing an evaluation set and far cheaper than simulation.
    """
    if n_pos < 1 or n_neg < 1:
        return float("nan")
    q1 = auc / (2.0 - auc)
    q2 = 2.0 * auc ** 2 / (1.0 + auc)
    variance = (
        auc * (1 - auc)
        + (n_pos - 1) * (q1 - auc ** 2)
        + (n_neg - 1) * (q2 - auc ** 2)
    ) / (n_pos * n_neg)
    return math.sqrt(max(variance, 0.0))


ASSUMED_AUC = 0.80

leverage = pd.DataFrame({
    "positives": gold[LABELS].sum().astype(int),
    "negatives": (len(gold) - gold[LABELS].sum()).astype(int),
})
leverage["se_at_0.80"] = [
    hanley_mcneil_se(ASSUMED_AUC, row.positives, row.negatives)
    for row in leverage.itertuples()
]
leverage["ci95_halfwidth"] = 1.96 * leverage["se_at_0.80"]
leverage = leverage.sort_values("se_at_0.80", ascending=False)

print(f"Per-label AUC uncertainty on the gold cohort (n={len(gold)}), assuming true AUC = {ASSUMED_AUC}")
print("-" * 72)
print(f"  {'label':18s} {'pos':>4s} {'neg':>4s} {'SE':>8s} {'95% CI half-width':>19s}")
for label, row in leverage.iterrows():
    print(f"  {label:18s} {int(row['positives']):>4d} {int(row['negatives']):>4d} "
          f"{row['se_at_0.80']:>8.3f} {row['ci95_halfwidth']:>18.3f}")

macro_se_independent = math.sqrt((leverage["se_at_0.80"] ** 2).sum()) / len(LABELS)
print()
print(f"Macro-AUC SE if labels were independent: {macro_se_independent:.4f} "
      f"(95% CI half-width {1.96 * macro_se_independent:.4f})")
print("Labels are NOT independent (section 2), so the true macro SE is larger than this.")

### Finding: at n=58, a single label's AUC is uncertain to about plus or minus 0.11 to 0.18

Even granting a genuinely good model (true AUC 0.80), the 95% confidence interval on a single
label spans **0.22 to 0.36 AUC points**. The rarest labels are the worst: `MCL`, with 9 positives,
carries a half-width of 0.18, and it counts exactly as much toward the final score as `Effusion`
with 35 positives and a half-width of 0.11.

The macro average is better behaved, since averaging twelve estimates cancels some noise — but
even the optimistic independence assumption leaves a half-width of roughly 0.04, and the labels
are demonstrably correlated, so the real figure is worse. **A 0.02 macro-AUC difference between
two models is invisible at this sample size.**

The practical consequence is a ranking of where effort pays: the low-positive labels (`MCL`,
`Lateral OA`, `Baker's`) dominate score variance while being the hardest to validate locally.

In [ ]:
def bootstrap_positive_counts(binary_frame, n_boot, rng):
    """Resample whole studies with replacement and record each label's positive count.

    Studies (not labels) are resampled so the co-occurrence structure from section 2 survives.
    """
    values = binary_frame.to_numpy(dtype=int)
    n_rows = values.shape[0]
    counts = np.empty((n_boot, values.shape[1]), dtype=int)
    for b in range(n_boot):
        idx = rng.integers(0, n_rows, size=n_rows)
        counts[b] = values[idx].sum(axis=0)
    return counts


def fold_degeneracy(binary_frame, n_folds, n_trials, rng):
    """Fraction of random k-fold splits in which some validation fold has zero positives.

    This is the practical question: not whether the whole cohort is scoreable, but whether a
    held-out fold is. AUC is undefined on a fold containing only negatives.
    """
    values = binary_frame.to_numpy(dtype=int)
    n_rows, n_labels = values.shape
    hit = np.zeros(n_labels, dtype=int)
    for _ in range(n_trials):
        order = rng.permutation(n_rows)
        bad = np.zeros(n_labels, dtype=bool)
        for fold in np.array_split(order, n_folds):
            bad |= values[fold].sum(axis=0) == 0
        hit += bad
    return hit / n_trials


boot_counts = bootstrap_positive_counts(gold[LABELS], N_BOOTSTRAP, RNG)
degenerate_fold_rate = fold_degeneracy(gold[LABELS], N_FOLDS, N_BOOTSTRAP, RNG)

summary = pd.DataFrame({
    "observed_pos": gold[LABELS].sum().astype(int),
    "boot_pos_p05": np.percentile(boot_counts, 5, axis=0).astype(int),
    "boot_pos_p95": np.percentile(boot_counts, 95, axis=0).astype(int),
    "pos_per_fold": gold[LABELS].sum().to_numpy() / N_FOLDS,
    "p_dead_fold": degenerate_fold_rate,
}, index=LABELS).sort_values("p_dead_fold", ascending=False)

print(f"Resampling the gold cohort (n={len(gold)}), {N_BOOTSTRAP:,} trials")
print("-" * 78)
print(f"  {'label':18s} {'pos':>4s} {'5th-95th pct':>14s} {'pos/fold':>9s} {'P(dead fold)':>14s}")
for label, row in summary.iterrows():
    print(f"  {label:18s} {int(row['observed_pos']):>4d} "
          f"{int(row['boot_pos_p05']):>6d}-{int(row['boot_pos_p95']):<7d} "
          f"{row['pos_per_fold']:>9.1f} {row['p_dead_fold']:>13.1%}")

swing = int(np.median(summary["boot_pos_p95"] - summary["boot_pos_p05"]) / 2)
print()
print(f"Positive counts swing by roughly +/-{swing} under resampling, on a base of {len(gold)} studies.")
print(f"With {N_FOLDS}-fold CV each validation fold holds about {len(gold) / N_FOLDS:.0f} studies.")
print(f"Labels at risk of an unscoreable fold: "
      f"{int((summary['p_dead_fold'] > 0.01).sum())} of {len(LABELS)}.")

### Finding: the gold 58 is an audit set, not a model-selection set

The analytic standard errors put per-label 95% intervals at roughly a quarter of the AUC range,
and resampling shows positive counts moving by about six studies in either direction on a base of
58.

The full cohort always stays scoreable — even `MCL` at 9 positives never loses all of them under
resampling — so the problem is not that AUC becomes undefined on the whole set. It appears one
level down. Five-fold cross-validation leaves about **12 studies per validation fold**, and at
that size the sparse labels start producing folds with no positives at all, where AUC is
undefined. The printed `P(dead fold)` column quantifies exactly which labels are exposed.

So use the gold cohort for what it can support: **measuring whether a report-based label extractor
agrees with radiologist ground truth** (section 5). That is a per-study agreement question where
58 examples is thin but usable, unlike a 0.01-AUC model comparison where it is hopeless.

For model selection, validate against **report-derived labels on the 4,349** — noisier per label,
but with 75 times the sample size. Trust the public leaderboard over local gold-58 CV, and expect
the leaderboard itself to be noisy given roughly 1,300 test studies.

## 4. The reports

Section 2 established that 4,349 studies have no labels and section 3 established that 58 studies
cannot train or validate a model. The reports are therefore the only route to a usable training
set, which makes their properties a first-order concern rather than a curiosity.

Three questions decide how hard that route is. **Is coverage complete** — does every study
actually have text? **What language is the text in** — the host says reports "may be in any of
several languages", and a monolingual English extractor would silently fail on the rest. And
**are any reports duplicated** — identical text across studies suggests either repeated
boilerplate or the same patient appearing twice, and either way those studies must not be split
across training and validation.

In [ ]:
reports = train["Report"].fillna("")
has_text = reports.str.strip().str.len() > 0

char_len = reports.str.len()
word_len = reports.str.split().str.len().fillna(0).astype(int)

print(f"Report coverage (all {len(train):,} training studies)")
print("-" * 58)
print(f"  non-null                      {train['Report'].notna().sum():>6,}")
print(f"  non-empty after strip         {int(has_text.sum()):>6,}  ({has_text.mean():.2%})")
print()
print("Length distribution (characters / words)")
for q in [0.01, 0.25, 0.50, 0.75, 0.99]:
    print(f"  p{int(q * 100):<3d}  {char_len.quantile(q):>8,.0f} chars   {word_len.quantile(q):>6,.0f} words")
print(f"  min   {char_len.min():>8,.0f} chars   {word_len.min():>6,.0f} words")
print(f"  max   {char_len.max():>8,.0f} chars   {word_len.max():>6,.0f} words")

### Detecting the language

We avoid a heavyweight language-detection dependency, both because the scoring environment has no
internet and because the problem here is easier than general language ID: reports are long enough
to vote on, and we only need the language well enough to pick keyword and negation vocabularies.

The approach is two-stage and deliberately conservative. Some scripts are unambiguous — text
written in Greek, Cyrillic, CJK, Arabic, Hebrew or Thai characters is settled by the script alone.
For Latin-script text we fall back to a stopword vote, scoring each candidate language by how many
of its distinctive function words appear. Anything without a clear winner is reported as
`Unresolved` rather than guessed, because a wrong language assignment would apply the wrong
negation cues in section 5 and flip labels.

In [ ]:
SCRIPT_RANGES = [
    ("Greek", 0x0370, 0x03FF), ("Greek", 0x1F00, 0x1FFF),
    ("Cyrillic", 0x0400, 0x04FF),
    ("Arabic", 0x0600, 0x06FF),
    ("Hebrew", 0x0590, 0x05FF),
    ("Thai", 0x0E00, 0x0E7F),
    ("CJK", 0x4E00, 0x9FFF), ("CJK", 0x3040, 0x30FF), ("CJK", 0xAC00, 0xD7AF),
]

# Distinctive function words, chosen to overlap as little as possible between languages.
# Compared after accent stripping, so they are written unaccented.
STOPWORDS = {
    "English": {"the", "and", "with", "there", "is", "no", "of", "normal", "left", "right", "knee"},
    "Spanish": {"resultados","técnica","el", "la", "los", "las", "con", "sin", "una", "del", "que", "rodilla", "se"},
    "Portuguese": {"do", "da", "dos", "das", "com", "sem", "uma", "que", "joelho", "nao"},
    "French": {"le", "les", "des", "avec", "sans", "une", "du", "genou", "pas", "est"},
    "German": {"reizödem","inlichtingen","kollateralbänder ", "der", "die", "das", "und", "mit", "ohne", "eine", "kein", "knie", "ist"},
    "Italian": {"il", "lo", "gli", "con", "senza", "una", "del", "ginocchio", "non", "che"},
    "Dutch": {"de", "het", "een", "met", "zonder", "geen", "van", "knie","niet", "is", "en", "op", "in", "ook", "tot", "als", "voor",    "aan", "bij", "zijn"},
    "Turkish": {"Diz eklemi","eklemi","ve", "ile", "bir", "yok", "var", "diz", "sol", "sag", "olarak", "izlendi"},
    "Polish": {"nie", "oraz", "jest", "sie", "kolana", "bez", "widoczne", "prawego", "lewego"},
    "Croatian": {"koljena", "zgloba", "hrskavice", "prati", "vidljiva", "nema", "uredan",
                 "uredna", "uredne", "meniskusa", "zglobnih", "nalazi", "presjeka",
                 "straznjeg", "prednjeg", "desnog", "lijevog"},
}

TOKEN_RE = re.compile(r"[a-z]+")            # Latin-only, for the stopword vote
UTOKEN_RE = re.compile(r"\w+", re.UNICODE)  # any script, for negation context


def dominant_script(text):
    """Return a non-Latin script name if it accounts for a meaningful share of the letters."""
    counts = Counter()
    letters = 0
    for ch in text:
        if not ch.isalpha():
            continue
        letters += 1
        code = ord(ch)
        for name, low, high in SCRIPT_RANGES:
            if low <= code <= high:
                counts[name] += 1
                break
    if letters == 0:
        return None
    script, count = (counts.most_common(1) or [(None, 0)])[0]
    return script if count / letters > 0.25 else None


def strip_accents(text):
    """Remove combining marks. Latin accents collapse to ASCII; Greek/Cyrillic letters survive."""
    return "".join(
        ch for ch in unicodedata.normalize("NFKD", text) if not unicodedata.combining(ch)
    )


def detect_language(text):
    """Script block first, then a Latin-script stopword vote. Ties return 'Unresolved'."""
    script = dominant_script(text)
    if script is not None:
        return script

    tokens = set(TOKEN_RE.findall(strip_accents(text.lower())))
    if not tokens:
        return "Unresolved"

    scores = {lang: len(tokens & words) for lang, words in STOPWORDS.items()}
    ranked = sorted(scores.items(), key=lambda kv: kv[1], reverse=True)
    best, best_score = ranked[0]
    runner_up_score = ranked[1][1] if len(ranked) > 1 else 0

    # Require a real signal and a clear margin, otherwise admit we do not know.
    if best_score < 3 or best_score == runner_up_score:
        return "Unresolved"
    return best


train["language"] = [detect_language(t) for t in reports]

language_counts = train["language"].value_counts()
print(f"Detected report language (all {len(train):,} training studies)")
print("-" * 58)
for language, count in language_counts.items():
    print(f"  {language:14s} {count:>6,}  {count / len(train):>6.1%}")

english_share = (train["language"] == "English").mean()
unresolved_share = (train["language"] == "Unresolved").mean()
non_latin_share = train["language"].isin(["Greek", "Cyrillic", "Arabic", "Hebrew", "CJK", "Thai"]).mean()
print()
print(f"  English                {english_share:>6.1%}")
print(f"  non-English (resolved) {1 - english_share - unresolved_share:>6.1%}")
print(f"  unresolved             {unresolved_share:>6.1%}")
print(f"  non-Latin script       {non_latin_share:>6.1%}   <- unreachable by Latin keyword rules")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4.6))

top_languages = language_counts.head(10)
axes[0].barh(top_languages.index[::-1], top_languages.values[::-1], color="#55a868")
axes[0].set_xlabel("Studies")
axes[0].set_title(f"Only {english_share:.0%} of reports are in English (n={len(train):,} studies)")
for i, value in enumerate(top_languages.values[::-1]):
    axes[0].text(value + 20, i, f"{value:,}", va="center", fontsize=9)

axes[1].hist(word_len.clip(upper=word_len.quantile(0.99)), bins=60, color="#c44e52")
axes[1].set_xlabel("Words per report (clipped at the 99th percentile)")
axes[1].set_ylabel("Studies")
axes[1].set_title(f"Reports are short: median {word_len.median():.0f} words (n={len(train):,})")

fig.tight_layout()
plt.show()

print("Caption: language is inferred by script block and stopword vote, not a trained classifier;")
print("'Unresolved' means the vote had no clear winner and is deliberately not guessed.")

In [ ]:
# Show a short excerpt per language so the reader can see what the text actually looks like.
print("Sample excerpts by detected language")
print("=" * 78)
for language in language_counts.head(11).index:
    subset = train.loc[train["language"] == language, "Report"]
    if subset.empty:
        continue
    excerpt = " ".join(str(subset.iloc[0]).split())[:200]
    print(f"\n[{language}]  ({len(subset):,} studies)")
    print(f"  {excerpt}...")

### Finding: reports are complete and short, and 6 in 10 are not in English

Every one of the 4,407 training studies has report text, so no study is unreachable by the text
pathway. The reports are also **short — a median of 129 words** — which is encouraging: these are
focused diagnostic conclusions rather than long narratives, so a keyword-plus-negation approach is
at least plausible.

The critical result is the language mix. **English accounts for only about 39% of reports.** The
remainder spans Spanish, Turkish, Greek, German, Cyrillic-script languages, Dutch, French and
Croatian, with roughly 12% written in a non-Latin script. An extractor built on English keywords
would return "no finding" for six of every ten studies — not an error, just quietly wrong labels
across most of the training set, which a model would then learn as truth.

The excerpts also reveal the *style* of these reports, which matters more than it first appears.
They are structured enumerations that walk through every anatomical structure in turn: "Lateral
menisküs ve medial menisküs normaldir", "the meniscus is not torn and there is no focal
chondrosis". **The anatomy is named in essentially every report whether or not it is abnormal.**
Any rule that treats the mention of "medial meniscus" as evidence of a meniscal tear will fire on
nearly everything. Section 5 is built around this constraint.

### Duplicate reports

Identical report text across two studies means one of two things, and both matter. It could be
institutional boilerplate, in which case the text carries no study-specific signal. Or it could be
the same patient or examination appearing twice, in which case putting one copy in training and
the other in validation leaks the answer.

Normalization has to be Unicode-aware here. An earlier version of this cell stripped everything
outside `[a-z0-9]`, which silently reduced every Greek and Cyrillic report to an empty string and
then reported all of them as one giant duplicate group — a fake finding produced entirely by the
normalizer. We keep word characters in any script, and separately count reports that normalize to
nothing as a sanity check.

In [ ]:
def normalize_report(text):
    """Lowercase, drop accents and punctuation, collapse whitespace. Unicode-aware."""
    text = strip_accents(str(text).lower())
    text = re.sub(r"[^\w\s]", " ", text, flags=re.UNICODE).replace("_", " ")
    return re.sub(r"\s+", " ", text).strip()


train["report_norm"] = [normalize_report(t) for t in reports]

empty_after_norm = int((train["report_norm"].str.len() == 0).sum())
print(f"Reports that normalize to an empty string: {empty_after_norm}  (sanity check; should be 0)")
print()

group_sizes = train["report_norm"].value_counts()
duplicate_groups = group_sizes[group_sizes > 1]
studies_in_duplicate_groups = int(duplicate_groups.sum())

print(f"Duplicate report text (all {len(train):,} training studies)")
print("-" * 58)
print(f"  distinct normalized reports   {len(group_sizes):>6,}")
print(f"  groups with >1 study          {len(duplicate_groups):>6,}")
print(f"  studies inside such a group   {studies_in_duplicate_groups:>6,}  "
      f"({studies_in_duplicate_groups / len(train):.2%})")
if len(duplicate_groups):
    print(f"  largest group                 {int(duplicate_groups.iloc[0]):>6,} studies")
    print()
    print("  Largest duplicate groups:")
    for text, size in duplicate_groups.head(5).items():
        print(f"    {size:>4,} studies  |  {text[:88]}...")

gold_in_duplicates = int(train.loc[GOLD_MASK, "report_norm"].isin(duplicate_groups.index).sum())
print()
print(f"  gold studies affected         {gold_in_duplicates:>6,} of {N_GOLD}")

### Finding: about 7.5% of studies share report text with another study

There are 73 groups of studies with byte-identical normalized reports, covering **333 studies
(7.56%)**, the largest containing 37. Reading the samples, these are template reports — a Turkish
all-normal template, a Spanish "sin anomalías" template — rather than obvious patient duplicates.

That distinction does not actually change what we must do. Whether the cause is boilerplate or a
repeated patient, two studies with identical text will receive identical weak labels, so splitting
them across training and validation lets the model see the validation answer. **Section 10 treats
every duplicate group as a single unit that cannot be split across folds.**

## 5. Weak-label baseline and gold audit

This is the pivotal experiment. If report text can be turned into reliable labels, the training
set grows from 58 studies to 4,407 and the competition becomes tractable. If it cannot, everything
downstream is built on noise.

We build the *simplest defensible* extractor — multilingual keyword matching with negation — not
because it is the best available method, but because it is transparent, produces an interpretable
failure analysis, and sets a floor that any learned approach must beat to justify its complexity.

Two design decisions come directly from what section 4 showed about these reports.

**Anatomy mentions are not findings.** Because reports enumerate every structure, "medial
meniscus" appears in nearly all of them. So for the anatomical labels we require an *abnormality*
term (tear, rupture, Riss, yırtık) near the anatomy term, not merely the anatomy itself. For
labels that are themselves findings — effusion, Baker's cyst, fracture — the keyword is sufficient.

**Negation usually comes after the keyword, not before.** Radiology writes "the ACL is intact",
"menisküs normaldir", "ligamento cruzado anterior íntegro". A negation window looking only
backwards, which is the textbook approach, misses nearly all of it. We scan a window on **both
sides** of the match, bounded by sentence punctuation.

We restrict to six findings with reasonably stable vocabulary. The three osteoarthritis
compartments, synovitis and contusion are graded and descriptive rather than named, so keyword
rules cannot express them; omitting them is more honest than reporting a baseline we know is
broken.

In [ ]:
# Anatomy terms, needing a nearby abnormality word to count as a finding.
ANATOMY_PATTERNS = {
    "ACL": [
        r"\bacl\b", r"anterior cruciate", r"cruzado anterior", r"cruzado anterior",
        r"croise anterieur", r"vordere[sn]? kreuzband", r"kreuzband", r"crociato anteriore",
        r"voorste kruisband", r"\blca\b", r"\bvkb\b", r"on capraz", r"προσθι", r"χιαστ",
        r"predn\w* kriznog", r"predn\w* krizni",
        r"кръстна връзка", r"кръстните връзки", r"крестообразн",
    ],
    "Medial Meniscus": [
        r"medial meniscus", r"menisco medial", r"menisco interno", r"menisque interne",
        r"menisque medial", r"innenmeniskus", r"mediale meniscus", r"menisco mediale",
        r"ic menisk", r"medial menisk", r"medijaln\w* meniskus", r"medijalnog meniskusa",
        r"εσω μηνισκ", r"медиалн\w* менискус", r"внутренн\w* мениск", r"медиальн\w* мениск",
    ],
    "Lateral Meniscus": [
        r"lateral meniscus", r"menisco lateral", r"menisco externo", r"menisque externe",
        r"menisque lateral", r"aussenmeniskus", r"laterale meniscus", r"menisco laterale",
        r"dis menisk", r"lateral menisk", r"lateraln\w* meniskus", r"lateralnog meniskusa",
        r"εξω μηνισκ", r"латералн\w* менискус", r"наружн\w* мениск", r"латеральн\w* мениск",
    ],
}

# Words that turn an anatomy mention into a positive finding.
ABNORMALITY_TERMS = [
    r"tear", r"torn", r"rupture", r"ruptured", r"lesion", r"injur", r"disrupt", r"defect",
    r"rotura", r"roto", r"desgarro", r"ruptura", r"lesion",
    r"dechirure", r"rompu", r"fissure",
    r"riss", r"ruptur", r"laesion", r"lasion",
    r"lacerazione", r"rottura",
    r"scheur", r"ruptuur",
    r"yirtik", r"yirtigi", r"rupturu", r"lezyon",
    r"ruptur\w*", r"puknuce", r"ozljed",
    r"ρηξ", r"ρηγμ",
    r"руптура", r"скъсван", r"увреда", r"лезия", r"разкъсван", r"разрыв", r"повреждени",
]

# Labels where the keyword is itself the finding.
FINDING_PATTERNS = {
    "Effusion": [
        r"effusion", r"derrame", r"epanchement", r"erguss", r"versamento", r"vocht in",
        r"joint fluid", r"liquido articular", r"eklem sivisi", r"hydrops", r"izliv",
        r"συλλογη", r"ставен излив", r"излив", r"выпот", r"жидкост",
    ],
    "Baker's": [
        r"baker", r"popliteal cyst", r"quiste popliteo", r"kyste poplite",
        r"poplitealzyste", r"cisti poplitea", r"popliteale cyste", r"bakerzyste",
        r"poplitealna cista", r"κυστη baker", r"κυστη μπεικερ",
        r"бекеров\w* киста", r"киста на бейкер", r"киста бейкера",
    ],
    "Fracture": [
        r"fracture", r"fractura", r"fratura", r"fraktur", r"frattura", r"fractuur",
        r"kirik", r"prijelom", r"καταγμα", r"фрактура", r"счупван", r"перелом",
    ],
}

# Negation cues across every language detected in section 4, including the inflected forms that
# matter (Turkish 'normaldir', German 'unauffallig', Greek 'χωρις', Russian 'без').
NEGATION_CUES = {
    # English
    "no", "not", "without", "absent", "negative", "intact", "normal", "unremarkable", "denies",
    "none", "free", "preserved",
    # Spanish / Portuguese
    "sin", "ningun", "ninguna", "ausencia", "ausente", "integro", "integra", "intacto", "normales",
    "sem", "nao", "conservado", "conservada",
    # French
    "pas", "sans", "absence", "aucun", "aucune", "intacte", "normale", "continu",
    # German
    "kein", "keine", "keiner", "ohne", "unauffallig", "unauffallige", "intakt", "nicht", "regelrecht",
    # Italian
    "senza", "nessun", "nessuna", "assenza", "non", "regolare",
    # Dutch
    "geen", "zonder", "afwezig", "intact", "niet", "normaal",
    # Turkish
    "yok", "yoktur", "izlenmedi", "saptanmadi", "normaldir", "dogaldir", "dogal", "normaldi",
    "izlenmemistir", "mevcut degildir",
    # Polish / Croatian / Serbian
    "nie", "bez", "brak", "nema", "uredan", "uredna", "uredni", "nije", "ocuvan", "ocuvana",
    # Greek (accents stripped)
    "χωρις", "δεν", "ουδεν", "ελευθερ", "φυσιολογικ", "ακεραι",
    # Bulgarian / Russian. 'б о' is the abbreviation 'б.о.' (unremarkable) after normalization.
    "няма", "без", "не", "нет", "нормалн", "нормално", "запазен", "непроменен",
    "интактн", "сохранен",
}

CONTEXT_WINDOW = 8  # tokens either side of the match, clipped at sentence boundaries
SENTENCE_BREAKS = ".;:\n"


def sentence_bounds(text, start, end):
    """Character span of the sentence containing [start, end)."""
    left = max((text.rfind(ch, 0, start) for ch in SENTENCE_BREAKS), default=-1)
    right_candidates = [text.find(ch, end) for ch in SENTENCE_BREAKS]
    right_candidates = [pos for pos in right_candidates if pos != -1]
    right = min(right_candidates) if right_candidates else len(text)
    return left + 1, right


def context_tokens(text, start, end, window=CONTEXT_WINDOW):
    """Tokens within `window` positions either side of a match, inside the same sentence."""
    left, right = sentence_bounds(text, start, end)
    before = UTOKEN_RE.findall(text[left:start])[-window:]
    after = UTOKEN_RE.findall(text[end:right])[:window]
    return before + after


def is_negated(tokens):
    """True if any context token is, or begins with, a negation cue.

    Prefix matching handles agglutinative and inflected forms (Turkish 'normaldir', Greek
    'φυσιολογικος'). Cues shorter than four characters must match exactly, so that 'no' does not
    swallow 'nodule'.
    """
    for token in tokens:
        for cue in NEGATION_CUES:
            if token == cue or (len(cue) >= 4 and token.startswith(cue)):
                return True
    return False


def extract_anatomy_label(text, patterns, abnormality_res):
    """Positive when an anatomy mention sits near an abnormality word and is not negated."""
    for pattern in patterns:
        for match in re.finditer(pattern, text):
            tokens = context_tokens(text, match.start(), match.end())
            joined = " ".join(tokens)
            if any(term.search(joined) for term in abnormality_res) and not is_negated(tokens):
                return 1
    return 0


def extract_finding_label(text, patterns):
    """Positive when a finding keyword appears and is not negated."""
    for pattern in patterns:
        for match in re.finditer(pattern, text):
            if not is_negated(context_tokens(text, match.start(), match.end())):
                return 1
    return 0


ABNORMALITY_RES = [re.compile(term) for term in ABNORMALITY_TERMS]
TRACTABLE_LABELS = list(ANATOMY_PATTERNS.keys()) + list(FINDING_PATTERNS.keys())

normalized_reports = [normalize_report(t) for t in train["Report"]]

weak_predictions = pd.DataFrame(index=train.index)
for label, patterns in ANATOMY_PATTERNS.items():
    weak_predictions[label] = [
        extract_anatomy_label(text, patterns, ABNORMALITY_RES) for text in normalized_reports
    ]
for label, patterns in FINDING_PATTERNS.items():
    weak_predictions[label] = [
        extract_finding_label(text, patterns) for text in normalized_reports
    ]
weak_predictions = weak_predictions[TRACTABLE_LABELS]

print(f"Rule-based extractor fired on all {len(train):,} training studies")
print("-" * 64)
print(f"  {'label':18s} {'positives':>10s} {'rate':>8s}")
for label in TRACTABLE_LABELS:
    positives = int(weak_predictions[label].sum())
    print(f"  {label:18s} {positives:>10,} {positives / len(train):>8.1%}")

print()
print(f"Not attempted ({len(LABELS) - len(TRACTABLE_LABELS)} labels): "
      f"{', '.join(l for l in LABELS if l not in TRACTABLE_LABELS)}")
print("These are graded/descriptive findings without stable keyword forms across languages.")

### Does the extractor reach every language?

A rule engine can fail in a way that looks like a negative result: if no pattern matches a
language at all, every study in that language receives all-zero labels, which a downstream model
reads as "confirmed healthy" rather than "unknown". Before trusting the audit, we check the share
of studies with no rule firing at all, broken down by language.

In [ ]:
no_hits = weak_predictions.sum(axis=1) == 0
coverage = (
    pd.DataFrame({"language": train["language"], "no_hits": no_hits})
    .groupby("language")
    .agg(studies=("no_hits", "size"), silent=("no_hits", "sum"))
)
coverage["silent_rate"] = coverage["silent"] / coverage["studies"]
coverage = coverage.sort_values("studies", ascending=False)

print(f"Studies where no rule fired at all (all {len(train):,} training studies)")
print("-" * 64)
print(f"  {'language':14s} {'studies':>8s} {'silent':>8s} {'silent rate':>13s}")
for language, row in coverage.iterrows():
    flag = "   <-- blind spot" if row["silent_rate"] > 0.60 else ""
    print(f"  {language:14s} {int(row['studies']):>8,} {int(row['silent']):>8,} "
          f"{row['silent_rate']:>12.1%}{flag}")

print()
print(f"Overall silent rate: {no_hits.mean():.1%} of studies receive no positive weak label.")

### Auditing the extractor against radiologist ground truth

This is the one job the gold 58 is actually sized for. We are not comparing models to three
decimal places; we are asking whether a rule fires on the right studies, which is a per-study
agreement question.

We report sensitivity (of the studies a radiologist marked positive, how many did the rule catch)
and specificity (of the negatives, how many did it correctly leave alone), each with a **Wilson**
confidence interval. Wilson is used rather than the textbook normal approximation because at these
counts — sometimes fewer than ten positives — the normal approximation produces intervals running
past 0 or 1 and understates uncertainty exactly where honesty matters most.

In [ ]:
def wilson_interval(successes, total, z=1.96):
    """Wilson score interval for a binomial proportion.

    Correct for small n and near-boundary proportions, where the normal approximation fails.
    """
    if total == 0:
        return (float("nan"), float("nan"))
    phat = successes / total
    denominator = 1 + z ** 2 / total
    centre = (phat + z ** 2 / (2 * total)) / denominator
    margin = z * math.sqrt(phat * (1 - phat) / total + z ** 2 / (4 * total ** 2)) / denominator
    return (max(0.0, centre - margin), min(1.0, centre + margin))


def audit_predictions(truth_frame, predicted_frame, labels):
    """Confusion counts plus Wilson intervals for sensitivity and specificity."""
    rows = []
    n = len(truth_frame)
    for label in labels:
        truth = truth_frame[label].to_numpy(dtype=int)
        predicted = predicted_frame[label].to_numpy(dtype=int)

        tp = int(((truth == 1) & (predicted == 1)).sum())
        fn = int(((truth == 1) & (predicted == 0)).sum())
        fp = int(((truth == 0) & (predicted == 1)).sum())
        tn = int(((truth == 0) & (predicted == 0)).sum())

        sens_lo, sens_hi = wilson_interval(tp, tp + fn)
        spec_lo, spec_hi = wilson_interval(tn, tn + fp)
        rows.append({
            "label": label, "n_gold": n, "tp": tp, "fp": fp, "tn": tn, "fn": fn,
            "sensitivity": tp / (tp + fn) if (tp + fn) else float("nan"),
            "sensitivity_lo": sens_lo, "sensitivity_hi": sens_hi,
            "specificity": tn / (tn + fp) if (tn + fp) else float("nan"),
            "specificity_lo": spec_lo, "specificity_hi": spec_hi,
            "accuracy": (tp + tn) / n,
            "balanced_accuracy": 0.5 * ((tp / (tp + fn) if (tp + fn) else 0)
                                        + (tn / (tn + fp) if (tn + fp) else 0)),
        })
    return pd.DataFrame(rows)


gold_predictions = weak_predictions.loc[GOLD_MASK]
audit = audit_predictions(gold, gold_predictions, TRACTABLE_LABELS)

print(f"Weak-label extractor audited against the gold cohort (n={len(gold)})")
print("=" * 94)
print(f"  {'label':18s} {'TP':>3s} {'FP':>3s} {'TN':>3s} {'FN':>3s}  "
      f"{'sensitivity (95% CI)':>24s}  {'specificity (95% CI)':>24s} {'bal.acc':>8s}")
for row in audit.itertuples():
    sens = f"{row.sensitivity:.2f} [{row.sensitivity_lo:.2f}-{row.sensitivity_hi:.2f}]"
    spec = f"{row.specificity:.2f} [{row.specificity_lo:.2f}-{row.specificity_hi:.2f}]"
    print(f"  {row.label:18s} {row.tp:>3d} {row.fp:>3d} {row.tn:>3d} {row.fn:>3d}  "
          f"{sens:>24s}  {spec:>24s} {row.balanced_accuracy:>8.2f}")

audit_path = ARTIFACT_DIR / "weak_label_audit.csv"
audit.to_csv(audit_path, index=False)
print()
print(f"Mean balanced accuracy across {len(audit)} labels: {audit['balanced_accuracy'].mean():.3f} "
      f"(0.50 = coin flip)")
print(f"Written: {audit_path}")

In [ ]:
fig, ax = plt.subplots(figsize=(11, 5))

y = np.arange(len(audit))
offset = 0.18
for metric, colour, shift, name in [
    ("sensitivity", "#4c72b0", offset, "Sensitivity (catches true positives)"),
    ("specificity", "#dd8452", -offset, "Specificity (leaves negatives alone)"),
]:
    values = audit[metric].to_numpy()
    lo = audit[f"{metric}_lo"].to_numpy()
    hi = audit[f"{metric}_hi"].to_numpy()
    ax.errorbar(values, y + shift, xerr=[values - lo, hi - values], fmt="o",
                color=colour, capsize=4, label=name)

ax.set_yticks(y)
ax.set_yticklabels(audit["label"])
ax.set_xlim(-0.02, 1.02)
ax.axvline(0.5, color="grey", linestyle=":", linewidth=1, label="Coin flip")
ax.set_xlabel("Proportion (95% Wilson interval)")
ax.set_title(f"Rule-based extraction, audited against radiologist labels (gold, n={len(gold)})")
ax.legend(loc="lower left")
fig.tight_layout()
plt.show()

print(f"Caption: intervals are wide because they rest on {len(gold)} studies;")
print("the ranking between labels is more trustworthy than any single value.")

### Where the rules fail

Aggregate metrics say how well the extractor does. To improve it you need to see the sentences it
got wrong, so we print the specific reports behind the false positives and false negatives.

In [ ]:
ALL_PATTERNS = {**ANATOMY_PATTERNS, **FINDING_PATTERNS}


def matched_span(normalized, label, radius=90):
    """Return the text around the first pattern hit, so we can see what triggered the rule."""
    for pattern in ALL_PATTERNS[label]:
        match = re.search(pattern, normalized)
        if match:
            lo = max(0, match.start() - radius)
            hi = min(len(normalized), match.end() + radius)
            return pattern, "..." + normalized[lo:hi] + "..."
    return None, None


def show_errors(label, kind, limit=2):
    """Print the evidence behind false positives or false negatives on the gold cohort."""
    truth = gold[label].to_numpy(dtype=int)
    predicted = gold_predictions[label].to_numpy(dtype=int)
    mask = (truth == 0) & (predicted == 1) if kind == "FP" else (truth == 1) & (predicted == 0)
    selected = gold.loc[mask]

    print(f"\n  {kind} for {label}  ({int(mask.sum())} of {len(gold)} gold studies)")
    for index in selected.head(limit).index:
        # `gold` was copied in section 2, before the language column existed.
        language = train.loc[index, "language"]
        normalized = normalize_report(train.loc[index, "Report"])
        pattern, span = matched_span(normalized, label)
        if kind == "FP":
            print(f"    [{language}] fired on /{pattern}/")
            print(f"      {span}")
        elif pattern is None:
            print(f"    [{language}] no keyword present at all (vocabulary gap)")
            print(f"      ...{normalized[:150]}...")
        else:
            print(f"    [{language}] keyword /{pattern}/ found but suppressed by negation")
            print(f"      {span}")


print("Error inspection (gold cohort)")
print("=" * 94)
for label in ["Medial Meniscus", "Effusion"]:
    show_errors(label, "FP")
    show_errors(label, "FN")

### Estimating agreement at scale

The audit rests on 58 studies. One further check is available across all 4,407: compare the rate
at which the extractor fires on the weak cohort against the prevalence a radiologist assigned on
the gold cohort. These are different populations and should not match exactly — the gold cohort is
finding-enriched — but an order-of-magnitude gap points to a broken rule rather than sampling
noise.

In [ ]:
comparison = pd.DataFrame({
    "gold_prevalence": [gold[label].mean() for label in TRACTABLE_LABELS],
    "rule_rate_gold": [gold_predictions[label].mean() for label in TRACTABLE_LABELS],
    "rule_rate_weak": [weak_predictions.loc[WEAK_MASK, label].mean() for label in TRACTABLE_LABELS],
}, index=TRACTABLE_LABELS)
comparison["gold_minus_rule_weak"] = comparison["gold_prevalence"] - comparison["rule_rate_weak"]

print("Prevalence comparison across cohorts")
print("-" * 78)
print(f"  {'label':18s} {'gold truth':>12s} {'rule on gold':>14s} {'rule on weak':>14s} {'gap':>8s}")
for label, row in comparison.iterrows():
    print(f"  {label:18s} {row['gold_prevalence']:>11.1%} {row['rule_rate_gold']:>13.1%} "
          f"{row['rule_rate_weak']:>13.1%} {row['gold_minus_rule_weak']:>+8.1%}")

print()
print(f"Gold cohort n={N_GOLD}; weak cohort n={N_WEAK:,}. The gold cohort is finding-enriched")
print("(section 2), so the rule firing less often on the weak cohort is expected, not alarming.")